# 00_Spark_Setup_and_Data — Lazy vs Action, RDD vs DataFrame

**Created:** 2026-02-05 04:59:58

> This notebook teaches PySpark step‑by‑step using the tiny array `123456781`. Everything is explained like you're 5: small stories, simple words, and prints everywhere so you can *see* what happens.


## 0) Spark Setup (ELI5)

- Think of Spark like a **kitchen**. The **driver** is the chef that reads the recipe. The **executors** are helpers who actually chop, stir, and cook.
- We ask the chef to **plan** the recipe (lazy *transformations*), and we only **cook** when we shout "Serve!" (an *action*).

Run the next cell to make a tiny kitchen on your laptop (`local[*]`). If PySpark isn't installed, the cell will tell you how to install it.


In [ ]:

# SparkSession is the doorway to Spark SQL/DataFrames
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
    from pyspark.sql.window import Window
    spark = (SparkSession
             .builder
             .appName("ELI5-PySpark-Operators")
             .master("local[*]")  # use all local cores
             .getOrCreate())
    print("✅ Spark is ready:", spark.version)
except Exception as e:
    print("❌ PySpark not available in this environment.")
    print("Install with: pip install pyspark --quiet")
    raise



## 1) Our tiny toys (data)

We turn `123456781` into:
- An **RDD** of digits (low-level Spark collection), and
- A **DataFrame** with one column `x` (higher-level table‑like structure).

We also add a tiny **index** column to make sorting/join demos easier.


In [ ]:

# Our raw Python list and string
py_list = [1,2,3,4,5,6,7,8,1]
py_str  = "123456781"

# Create an RDD from the list
rdd = spark.sparkContext.parallelize(py_list, 2)  # 2 partitions so you can see partition behavior
print("RDD partitions:", rdd.getNumPartitions())
print("RDD sample:", rdd.take(5))

# Create a DataFrame with an index
df = spark.createDataFrame([(i, v) for i, v in enumerate(py_list)], schema=["idx","x"]) 
print("DataFrame schema:")
df.printSchema()
print("First rows:")
df.show(5, truncate=False)



### Lazy vs Action (peek)
- **Transformations** (e.g., `select`, `filter`) are like *planning* steps — Spark **remembers** them but **doesn't run** yet.
- **Actions** (e.g., `show`, `collect`) say **"Go cook now!"** and trigger execution.

Run the next cell to feel the difference.


In [ ]:

# Build a plan (lazy): select and filter — NOTHING runs yet
planned = df.select("x").where(F.col("x") > 3)
print("We created a plan. Has Spark executed? Not yet.")

# Now an action: show() triggers the plan to run now
print("
Calling show(): triggers execution")
planned.show()
